# ⚖️ Legal AI Assistant — RAG Pipeline
### Deep Learning End-Semester Project | Track A: RAG

This notebook demonstrates the full RAG pipeline:
1. Install dependencies
2. Load & chunk a legal document
3. Embed chunks with `sentence-transformers`
4. Store in ChromaDB
5. Query with an LLM (Groq free API or vLLM)
6. Launch the Streamlit app with `ngrok`

**Runtime:** GPU (T4 recommended) — Runtime > Change runtime type > T4 GPU

## 📦 Step 1 — Install Dependencies

In [ ]:
!pip install -q fastapi uvicorn langchain langchain-community chromadb \
    sentence-transformers openai pypdf2 python-multipart pydantic \
    streamlit requests pyngrok groq 2>/dev/null
print('✅ All packages installed')

## 📂 Step 2 — Clone the GitHub Repository

In [ ]:
import os

# ── CHANGE THIS to your GitHub repo URL ──────────────────────────────────────
GITHUB_REPO = 'https://github.com/YOUR_USERNAME/dl-project.git'
# ─────────────────────────────────────────────────────────────────────────────

if not os.path.exists('/content/dl-project'):
    !git clone {GITHUB_REPO} /content/dl-project
else:
    !cd /content/dl-project && git pull

os.chdir('/content/dl-project')
print('✅ Repository ready at /content/dl-project')

## 🔑 Step 3 — Configure LLM API Key

**Option A:** Use **Groq** (free, fast — recommended for demo)  
Get a free key at https://console.groq.com → API Keys

**Option B:** Use **OpenAI** API key

**Option C:** Deploy vLLM on RunPod/Vast.ai and paste the endpoint URL

In [ ]:
import os

# ── OPTION A: Groq (free tier) ────────────────────────────────────────────────
GROQ_API_KEY = 'gsk_YOUR_GROQ_KEY_HERE'   # ← paste your Groq key

os.environ['LLM_API_URL']  = 'https://api.groq.com/openai/v1'
os.environ['LLM_API_KEY']  = GROQ_API_KEY
os.environ['MODEL_NAME']   = 'llama3-8b-8192'   # fast Groq-hosted LLaMA 3

# ── OPTION B: OpenAI ──────────────────────────────────────────────────────────
# os.environ['LLM_API_URL'] = 'https://api.openai.com/v1'
# os.environ['LLM_API_KEY'] = 'sk-YOUR_OPENAI_KEY'
# os.environ['MODEL_NAME']  = 'gpt-3.5-turbo'

# ── OPTION C: vLLM on RunPod ──────────────────────────────────────────────────
# os.environ['LLM_API_URL'] = 'https://YOUR-RUNPOD-ID-8000.proxy.runpod.net/v1'
# os.environ['LLM_API_KEY'] = 'dummy-key'
# os.environ['MODEL_NAME']  = 'meta-llama/Llama-3.2-3B-Instruct'

print(f'✅ LLM configured: {os.environ["MODEL_NAME"]} @ {os.environ["LLM_API_URL"]}')

## 🧪 Step 4 — Test the RAG Pipeline Directly

In [ ]:
import sys
sys.path.insert(0, '/content/dl-project')

from app.rag_pipeline import RAGPipeline

rag = RAGPipeline()
print('✅ RAG pipeline initialized')

# Load the sample legal document
with open('sample_docs/sample_legal.txt', 'r') as f:
    text = f.read()

rag.add_document(text, 'sample_legal.txt')
print(f'✅ Document loaded — {len(text)} characters')

In [ ]:
# Ask a question
question = 'What are the termination conditions in this contract?'

result = rag.query(question)

print('=' * 60)
print(f'Q: {question}')
print('=' * 60)
print(f'A: {result["answer"]}')
print()
print(f'📚 Sources: {len(result["sources"])} chunks retrieved')
for i, s in enumerate(result['sources'], 1):
    print(f'  [{i}] {s["filename"]} — chunk {s["chunk_index"]}')
    print(f'      {s["content"][:120]}…')

## 📄 Step 5 — Upload Your Own Document (Optional)

In [ ]:
from google.colab import files
import io, PyPDF2

uploaded = files.upload()  # Opens file picker

for filename, content in uploaded.items():
    if filename.endswith('.pdf'):
        reader = PyPDF2.PdfReader(io.BytesIO(content))
        text = '\n'.join(p.extract_text() or '' for p in reader.pages)
    else:
        text = content.decode('utf-8')
    
    rag.add_document(text, filename)
    print(f'✅ Uploaded and indexed: {filename} ({len(text)} chars)')

## 🌐 Step 6 — Launch Streamlit App with ngrok (Public URL)

Get a free ngrok token at https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
from pyngrok import ngrok, conf
import subprocess, threading, time

# ── Paste your ngrok auth token here ─────────────────────────────────────────
NGROK_TOKEN = 'YOUR_NGROK_TOKEN_HERE'   # ← get free token at ngrok.com
# ─────────────────────────────────────────────────────────────────────────────

ngrok.set_auth_token(NGROK_TOKEN)

# Start FastAPI backend in background
backend = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8080'],
    cwd='/content/dl-project'
)
time.sleep(3)
print('✅ FastAPI backend started')

# Start Streamlit
streamlit_proc = subprocess.Popen(
    ['streamlit', 'run', 'app/streamlit_app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false'],
    cwd='/content/dl-project',
    env={**__import__('os').environ, 'API_URL': 'http://localhost:8080'}
)
time.sleep(4)
print('✅ Streamlit started')

# Open ngrok tunnel
public_url = ngrok.connect(8501)
print()
print('=' * 60)
print(f'🌐 PUBLIC URL: {public_url}')
print('=' * 60)
print('Share this link with your instructor to demo the live app!')

## 🧹 Step 7 — Stop All Processes

In [ ]:
ngrok.kill()
backend.terminate()
streamlit_proc.terminate()
print('✅ All processes stopped')

---
## 📊 Evaluation — Retrieval Quality Metrics

In [ ]:
# Quick evaluation on sample questions
test_questions = [
    'What is the notice period for termination?',
    'What confidential information is covered?',
    'What is the governing law?',
    'What are the employee responsibilities?',
    'What are the non-compete restrictions?',
]

print('=' * 60)
print('EVALUATION — RAG Pipeline Test')
print('=' * 60)

for i, q in enumerate(test_questions, 1):
    result = rag.query(q)
    n_sources = len(result['sources'])
    has_answer = len(result['answer']) > 20
    print(f'[{i}] Q: {q[:55]}...')
    print(f'     Sources: {n_sources} | Has answer: {"✅" if has_answer else "❌"}')
    print(f'     A: {result["answer"][:100]}...')
    print()

print('✅ Evaluation complete')